In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

In [ ]:
def parse_kml_points(kml_path):
    tree = ET.parse(kml_path)
    root = tree.getroot()
    rows = []

    if root.tag.startswith("{"):
        ns = root.tag[:root.tag.find("}") + 1]
    else:
        ns = ""

    def add_coords(coord_text, meta):
        """Parse one <coordinates> block, add one row per point with metadata."""
        for part in coord_text.replace("\n", " ").split():
            if not part.strip():
                continue
            pieces = part.split(",")
            if len(pieces) >= 2:
                lon = float(pieces[0])
                lat = float(pieces[1])
                alt = float(pieces[2]) if len(pieces) >= 3 else None

                row = meta.copy()
                row["lat"] = lat
                row["lon"] = lon
                row["alt_kml"] = alt
                rows.append(row)

    # iterate over all Placemarks (they carry the ExtendedData)
    for pm in root.iter(f"{ns}Placemark"):
        meta = {}

        # optional: name / description / timestamp
        name_el = pm.find(f"{ns}name")
        if name_el is not None:
            meta["name"] = name_el.text

        desc_el = pm.find(f"{ns}description")
        if desc_el is not None:
            meta["description"] = desc_el.text

        when_el = pm.find(f".//{ns}when")
        if when_el is not None:
            meta["timestamp"] = when_el.text

        # IMPORTANT: ExtendedData / SchemaData / SimpleData
        # <SimpleData name="Speed">98</SimpleData>  -> meta["Speed"] = "98"
        for sd in pm.findall(f".//{ns}SimpleData"):
            key = sd.attrib.get("name")
            if key:
                meta[key] = sd.text

        # geometries: Points (your MovementReport) and also LineStrings if present
        for point in pm.findall(f".//{ns}Point"):
            coord_el = point.find(f"{ns}coordinates")
            if coord_el is not None and coord_el.text:
                add_coords(coord_el.text, meta)

        for ls in pm.findall(f".//{ns}LineString"):
            coord_el = ls.find(f"{ns}coordinates")
            if coord_el is not None and coord_el.text:
                add_coords(coord_el.text, meta)

    return rows


def main():
    rows = parse_kml_points(INPUT_KML)
    print(f"Extracted {len(rows)} points from {INPUT_KML}")

    # rows is list of dicts with lat/lon/alt_kml + SimpleData/meta columns
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved CSV: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()